### Importing Libraries

In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
import matplotlib.pyplot as plt

### Load the dataset

In [2]:
df = pd.read_csv("StudentPerformanceFactors.csv")

df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


### Data Cleaning (removing hidden spaces)

In [3]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

### Encoding (Ordinal Variables)

In [4]:
ordinal_maps = {
    'Parental_Involvement': {'Low':0, 'Medium':1, 'High':2},
    'Motivation_Level': {'Low':0, 'Medium':1, 'High':2},
    'Family_Income': {'Low':0, 'Medium':1, 'High':2},
    'Teacher_Quality': {'Low':0, 'Medium':1, 'High':2},
    'Parental_Education_Level': {'High School':0, 'College':1, 'Postgraduate':2},
    'Peer_Influence': {'Negative':0, 'Neutral':1, 'Positive':2},
    'Distance_from_Home': {'Near':0, 'Moderate':1, 'Far':2},
    'Access_to_Resources': {'Low':0, 'Medium':1, 'High':2},
    'Extracurricular_Activities': {'No':0, 'Yes':1},
    'Internet_Access': {'No':0, 'Yes':1},
    'Learning_Disabilities': {'No':0, 'Yes':1}
}

for col, mapping in ordinal_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

### Encoding (Nominal Variables)

In [5]:
df = pd.get_dummies(df, columns=['Gender', 'School_Type'], drop_first=True)

In [6]:
# converting the boolean to integers
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

In [7]:
# converting everything to numeric values
df = df.apply(pd.to_numeric, errors='coerce')

In [8]:
# getting rid of missing values
df = df.dropna()

In [9]:
df.shape

(6378, 20)

In [10]:
print("Missing values:", df.isnull().sum().sum())
print("Object columns:", df.select_dtypes(include='object').columns)

Missing values: 0
Object columns: Index([], dtype='object')


In [11]:
df

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Exam_Score,Gender_Male,School_Type_Public
0,23,84,0,2,0,7,73,0,1,0,0,1.0,2,3,0,0.0,0.0,67,1,1
1,19,64,0,1,0,8,59,0,1,2,1,1.0,0,4,0,1.0,1.0,61,0,1
2,24,98,1,1,1,7,91,1,1,2,1,1.0,1,4,0,2.0,0.0,74,1,1
3,29,89,0,1,1,8,98,1,1,1,1,1.0,0,4,0,0.0,1.0,71,1,1
4,19,92,1,1,1,6,65,1,1,3,1,2.0,1,4,0,1.0,0.0,70,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6602,25,69,2,1,0,7,76,1,1,1,2,1.0,2,2,0,0.0,0.0,68,0,1
6603,23,76,2,1,0,8,81,1,1,3,0,2.0,2,2,0,0.0,0.0,69,0,1
6604,20,90,1,0,1,6,65,0,1,3,0,1.0,0,2,0,2.0,0.0,68,0,1
6605,10,86,2,2,1,6,91,2,1,2,0,1.0,2,3,0,0.0,2.0,68,0,0


### OLS Regression

In [12]:
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:             Exam_Score   R-squared:                       0.721
Model:                            OLS   Adj. R-squared:                  0.721
Method:                 Least Squares   F-statistic:                     866.8
Date:                Fri, 20 Mar 2026   Prob (F-statistic):               0.00
Time:                        08:58:16   Log-Likelihood:                -13677.
No. Observations:                6378   AIC:                         2.739e+04
Df Residuals:                    6358   BIC:                         2.753e+04
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               

##### REMARKS:

###### sleep hours, gender and school type are not significant since the p values >0.05. They do not have a significant effect on exam score.
###### The other variables are statistically significant since their p values<0.05

###### R-squared shows the goodness of fit of the model. 0.721 means that 72.1% of variation in exam scores is explained by the variables in the model

###### F-statistic tests whether the whole model is useful. The corresponding p value of 0.00 indicates that the model is statistically significant since it is less than 0.05. The independent variables jointly have a significant effect on exam scores.

###### Durbin-Watson=2, this means there is no autocorrelation problem



### Test for Multicollinearity

In [13]:
vif_data = pd.DataFrame({
    "Variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

print(vif_data)

                      Variable         VIF
0                        const  161.849123
1                Hours_Studied    1.002064
2                   Attendance    1.004210
3         Parental_Involvement    1.004267
4          Access_to_Resources    1.003600
5   Extracurricular_Activities    1.001840
6                  Sleep_Hours    1.002749
7              Previous_Scores    1.003785
8             Motivation_Level    1.002322
9              Internet_Access    1.002431
10           Tutoring_Sessions    1.001602
11               Family_Income    1.002318
12             Teacher_Quality    1.001424
13              Peer_Influence    1.002353
14           Physical_Activity    1.003627
15       Learning_Disabilities    1.002310
16    Parental_Education_Level    1.002772
17          Distance_from_Home    1.001881
18                 Gender_Male    1.001659
19          School_Type_Public    1.002571


###### REMARK: since all the Variance Inflation Factor (VIF) values are approximately 1, there is no multicollinearity problem 

### Test for Heteroskedasticity

In [14]:
bp_test = het_breuschpagan(model.resid, model.model.exog)

labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
print(dict(zip(labels, bp_test)))

{'LM Statistic': np.float64(14.809513284327826), 'LM-Test p-value': np.float64(0.7346249453354535), 'F-Statistic': np.float64(0.7788122678588065), 'F-Test p-value': np.float64(0.7352219248340193)}


###### REMARK: both the LM test(0.7346) and F-test(0.7352) p values are >0,05, the model does not suffer from heteroskedasticity, the variance of errors is constant

##### Saving the df as csv

In [15]:
df.to_csv("student_data.csv", index=False)